# 단계 1: 애널리스트 리포트 수집 및 OCR 처리

**목표**: Naver Finance에서 리포트를 크롤링 → PDF 다운로드 → 텍스트 변환 → 한글 정제

**출력**: `data/raw/[기업명]_report_text.xlsx`

## 1. 필수 라이브러리 설치 및 임포트

In [ ]:
# 필수 라이브러리 설치
!pip install -q pandas requests beautifulsoup4 lxml pdfminer.six openpyxl pyyaml

In [ ]:
import sys
import os

# src 모듈 경로 추가
sys.path.insert(0, os.path.abspath('../..'))

from src.crawler import naver_crawler, pdf_download
from src.ocr_processor import pdfread, extract_txt
import pandas as pd
import yaml
from datetime import datetime

print("✅ 모든 라이브러리 임포트 완료")

## 2. 설정 로드

In [ ]:
# config.yaml 로드
config_path = os.path.join(os.path.dirname(__file__), '..', 'config.yaml')

with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print("✅ 설정 로드 완료")
print(f"시작 날짜: {config['CRAWLER_CONFIG']['start_date']}")
print(f"종료 날짜: {config['CRAWLER_CONFIG']['end_date']}")

## 3. 파라미터 설정

In [ ]:
# ⚙️ 사용자 설정
COMPANY_NAME = '삼성전자'  # 크롤링할 기업명 (변경 가능)
START_DATE = '20240101'
END_DATE = '20241231'

# 데이터 경로
base_path = os.path.join(config['DATA_PATHS']['raw_data'], COMPANY_NAME)
pdf_path = os.path.join(base_path, 'pdf')
txt_path = os.path.join(base_path, 'txt')

# 디렉토리 생성
os.makedirs(base_path, exist_ok=True)
os.makedirs(pdf_path, exist_ok=True)
os.makedirs(txt_path, exist_ok=True)

print(f"✅ 디렉토리 준비 완료")
print(f"   기업: {COMPANY_NAME}")
print(f"   저장 경로: {base_path}")

## 4. 리포트 크롤링

In [ ]:
# Naver Finance에서 리포트 크롤링
print(f"\n🔄 {COMPANY_NAME} 리포트 크롤링 시작...")

try:
    df_report = naver_crawler(COMPANY_NAME, START_DATE, END_DATE)
    print(f"\n✅ 크롤링 완료: {len(df_report)}개 리포트")
    print(f"\n컬럼: {df_report.columns.tolist()}")
    df_report.head(3)
except Exception as e:
    print(f"❌ 크롤링 실패: {e}")

## 5. PDF 다운로드

In [ ]:
# PDF 다운로드
print(f"\n🔄 PDF 다운로드 시작...")

try:
    pdf_download(df_report['pdf'], pdf_path)
    print(f"\n✅ PDF 다운로드 완료")
    
    # 다운로드된 파일 확인
    pdf_files = [f for f in os.listdir(pdf_path) if f.endswith('.pdf')]
    print(f"   저장된 PDF 파일 수: {len(pdf_files)}")
except Exception as e:
    print(f"❌ PDF 다운로드 실패: {e}")

## 6. PDF를 텍스트로 변환

In [ ]:
# PDF 읽기 및 텍스트 저장
print(f"\n🔄 PDF → 텍스트 변환 중...")

try:
    pdfread(pdf_path, txt_path)
    print(f"\n✅ 텍스트 변환 완료")
    
    # 변환된 파일 확인
    txt_files = [f for f in os.listdir(txt_path) if f.endswith('.txt')]
    print(f"   저장된 텍스트 파일 수: {len(txt_files)}")
except Exception as e:
    print(f"❌ 텍스트 변환 실패: {e}")

## 7. 텍스트 정제 (한글만 추출)

In [ ]:
# 텍스트 정제
print(f"\n🔄 텍스트 정제 중 (한글 추출)...")

try:
    df_txt = extract_txt(txt_path)
    print(f"\n✅ 텍스트 정제 완료")
    print(f"   처리된 문서 수: {len(df_txt)}")
    
    # 샘플 데이터 확인
    if len(df_txt) > 0:
        print(f"\n첫 번째 문서 샘플 (처음 500자):")
        print(df_txt.iloc[0]['text'][:500])
except Exception as e:
    print(f"❌ 텍스트 정제 실패: {e}")

## 8. 데이터 병합 및 저장

In [ ]:
# 메타데이터와 텍스트 병합
print(f"\n🔄 데이터 병합 및 저장 중...")

try:
    df_merged = pd.merge(df_report, df_txt, on='pdf', how='left')
    
    # Excel 저장
    output_excel = os.path.join(base_path, f"{COMPANY_NAME}_report_text.xlsx")
    df_merged.to_excel(output_excel, index=False)
    
    # CSV 저장
    output_csv = os.path.join(base_path, f"{COMPANY_NAME}_report_text.csv")
    df_merged.to_csv(output_csv, index=False, encoding='utf-8-sig')
    
    print(f"\n✅ 저장 완료")
    print(f"   Excel: {output_excel}")
    print(f"   CSV: {output_csv}")
    print(f"\n최종 데이터:")
    print(f"   행 수: {len(df_merged)}")
    print(f"   컬럼: {df_merged.columns.tolist()}")
    print(f"\n📊 데이터 구조:")
    print(df_merged.head(2))
except Exception as e:
    print(f"❌ 저장 실패: {e}")